# 📊 Module 12 Lab: Agent Evaluation & Continuous Improvement

**FinTech Corp · LoanBot v3.0 Evaluation Engineering**

| Section | Nội dung |
|---------|----------|
| 1 | Golden Dataset Builder — 100 labeled cases |
| 2 | LLM-as-Judge — evaluate RiskAgent outputs |
| 3 | Regression Suite — automated quality gates |
| 4 | A/B Test Analyzer — statistical significance |
| 5 | RAGAS Metrics — faithfulness, relevancy, precision, recall |
| 6 | Continuous Improvement Dashboard |

**Không cần API key thật** — toàn bộ simulation với mock functions.

In [ ]:
# Setup
from __future__ import annotations
from dataclasses import dataclass, field
from datetime import datetime
from typing import Dict, List, Optional, Tuple
from collections import defaultdict
import random
import json
import math
import statistics
import hashlib

random.seed(42)  # reproducible
print('✅ Module 12 Lab — LoanBot Evaluation Engineering')
print('📅', datetime.now().strftime('%Y-%m-%d %H:%M'))

---
## Section 1: Golden Dataset Builder

Tạo 100 hồ sơ golden với expert labels — ground truth cho evaluation.

In [ ]:
@dataclass
class GoldenCase:
    case_id: str
    credit_score: int
    loan_amount_vnd: int
    monthly_income_vnd: int
    dti_ratio: float
    fraud_probability: float
    is_self_employed: bool
    payment_history: str          # "CLEAN", "LATE_ONCE", "MULTIPLE_LATE"
    expert_label: str             # "APPROVE" | "REJECT"
    label_confidence: str         # "HIGH" | "MEDIUM" (borderline)
    actual_outcome: str           # "PAID" | "DEFAULT" — 12-month outcome
    tags: List[str]               # ["borderline", "self_employed", "high_amount"]


class GoldenDatasetBuilder:
    """Generates synthetic golden dataset for LoanBot evaluation."""

    def build(self, n: int = 100) -> List[GoldenCase]:
        cases = []
        # 40 APPROVE cases (credit score 680+, good DTI, clean history)
        for i in range(40):
            score  = random.randint(700, 820)
            income = random.randint(20_000_000, 50_000_000)
            loan   = random.randint(100_000_000, 800_000_000)
            dti    = round(random.uniform(0.15, 0.38), 2)
            self_emp = random.random() < 0.2
            tags = ['good_credit']
            if self_emp: tags.append('self_employed')
            if loan > 500_000_000: tags.append('high_amount')
            cases.append(GoldenCase(
                case_id=f'GLD-A-{i+1:03d}',
                credit_score=score, loan_amount_vnd=loan,
                monthly_income_vnd=income, dti_ratio=dti,
                fraud_probability=round(random.uniform(0.01, 0.08), 2),
                is_self_employed=self_emp,
                payment_history=random.choice(['CLEAN', 'CLEAN', 'LATE_ONCE']),
                expert_label='APPROVE', label_confidence='HIGH',
                actual_outcome='PAID', tags=tags
            ))

        # 40 REJECT cases (low score, high DTI, fraud signals, bad history)
        for i in range(40):
            score  = random.randint(350, 540)
            income = random.randint(5_000_000, 18_000_000)
            loan   = random.randint(50_000_000, 500_000_000)
            dti    = round(random.uniform(0.45, 0.75), 2)
            self_emp = random.random() < 0.35
            tags = ['poor_credit']
            if self_emp: tags.append('self_employed')
            fraud = round(random.uniform(0.15, 0.65), 2)
            if fraud > 0.3: tags.append('fraud_signal')
            cases.append(GoldenCase(
                case_id=f'GLD-R-{i+1:03d}',
                credit_score=score, loan_amount_vnd=loan,
                monthly_income_vnd=income, dti_ratio=dti,
                fraud_probability=fraud,
                is_self_employed=self_emp,
                payment_history=random.choice(['LATE_ONCE', 'MULTIPLE_LATE', 'MULTIPLE_LATE']),
                expert_label='REJECT', label_confidence='HIGH',
                actual_outcome='DEFAULT', tags=tags
            ))

        # 20 BORDERLINE cases (score 560–680, expert disagreement possible)
        for i in range(20):
            score  = random.randint(560, 680)
            income = random.randint(15_000_000, 30_000_000)
            loan   = random.randint(100_000_000, 400_000_000)
            dti    = round(random.uniform(0.35, 0.50), 2)
            self_emp = random.random() < 0.5
            label  = 'APPROVE' if random.random() > 0.45 else 'REJECT'
            outcome = ('PAID' if label == 'APPROVE' and random.random() > 0.2
                       else 'DEFAULT')
            tags = ['borderline']
            if self_emp: tags.append('self_employed')
            cases.append(GoldenCase(
                case_id=f'GLD-B-{i+1:03d}',
                credit_score=score, loan_amount_vnd=loan,
                monthly_income_vnd=income, dti_ratio=dti,
                fraud_probability=round(random.uniform(0.05, 0.20), 2),
                is_self_employed=self_emp,
                payment_history=random.choice(['CLEAN', 'LATE_ONCE']),
                expert_label=label, label_confidence='MEDIUM',
                actual_outcome=outcome, tags=tags
            ))

        return cases


builder   = GoldenDatasetBuilder()
golden_ds = builder.build(100)

# Summary
from collections import Counter
label_counts = Counter(c.expert_label for c in golden_ds)
tag_counts   = Counter(t for c in golden_ds for t in c.tags)
confidence   = Counter(c.label_confidence for c in golden_ds)

print(f"\n{'━'*50}")
print(f" Golden Dataset Summary ({len(golden_ds)} cases)")
print(f"{'─'*50}")
print(f" Labels:      {dict(label_counts)}")
print(f" Confidence:  {dict(confidence)}")
print(f" Tags:        {dict(tag_counts)}")
print(f"\n Sample (first 3):")
for c in golden_ds[:3]:
    print(f"  {c.case_id}: score={c.credit_score}, DTI={c.dti_ratio}, label={c.expert_label}, tags={c.tags}")

---
## Section 2: LLM-as-Judge — Evaluate RiskAgent Outputs

In [ ]:
@dataclass
class JudgeScore:
    case_id: str
    logical_consistency: float  # 1-5
    factor_coverage:     float
    confidence_calib:    float
    actionability:       float
    conciseness:         float
    weaknesses: List[str]

    @property
    def overall(self) -> float:
        return statistics.mean([
            self.logical_consistency, self.factor_coverage,
            self.confidence_calib, self.actionability, self.conciseness
        ])


class MockJudgeAgent:
    """Mocks Claude-as-judge scoring RiskAgent outputs.
    In production: calls Anthropic API with judge prompt.
    """

    def judge(self, case: GoldenCase, agent_output: dict) -> JudgeScore:
        # Simulate judge scoring based on case characteristics
        recommendation = agent_output.get('recommendation', 'REVIEW')
        confidence     = agent_output.get('confidence', 0.7)
        reasoning      = agent_output.get('reasoning', '')

        # Logical consistency: is recommendation aligned with case quality?
        expected = case.expert_label
        lc = 4.5 if recommendation == expected else 2.0
        lc += random.uniform(-0.3, 0.3)

        # Factor coverage: does reasoning mention key factors?
        key_factors = ['credit score', 'dti', 'income', 'fraud']
        mentioned   = sum(1 for f in key_factors if f.lower() in reasoning.lower())
        fc = 1.0 + mentioned * 1.0 + random.uniform(-0.2, 0.2)

        # Confidence calibration: overconfident for borderline cases?
        is_borderline = 'borderline' in case.tags
        if is_borderline and confidence > 0.80:
            cc = 2.5 + random.uniform(-0.5, 0.5)  # overconfident
        else:
            cc = 4.0 + random.uniform(-0.5, 0.5)

        # Actionability: is output useful for loan officer?
        ac = 4.2 + random.uniform(-0.4, 0.4)

        # Conciseness
        word_count = len(reasoning.split())
        cn = 4.0 if 30 <= word_count <= 120 else (3.0 if word_count > 120 else 2.5)
        cn += random.uniform(-0.2, 0.2)

        weaknesses = []
        if cc < 3.5:
            weaknesses.append('Confidence overestimated for borderline case')
        if fc < 3.0:
            weaknesses.append('Missing key factors in reasoning')
        if lc < 3.0:
            weaknesses.append('Recommendation inconsistent with evidence')

        return JudgeScore(
            case_id=case.case_id,
            logical_consistency=round(min(5.0, max(1.0, lc)), 1),
            factor_coverage    =round(min(5.0, max(1.0, fc)), 1),
            confidence_calib   =round(min(5.0, max(1.0, cc)), 1),
            actionability      =round(min(5.0, max(1.0, ac)), 1),
            conciseness        =round(min(5.0, max(1.0, cn)), 1),
            weaknesses=weaknesses
        )


def mock_risk_agent(case: GoldenCase) -> dict:
    """Simulates RiskAgent output with realistic characteristics."""
    score = case.credit_score
    dti   = case.dti_ratio
    fraud = case.fraud_probability

    # Decision logic (intentionally slightly miscalibrated for borderline)
    approve_score = (score - 400) / 450 - dti * 0.5 - fraud * 2
    recommendation = 'APPROVE' if approve_score > 0 else 'REJECT'

    # Confidence: overconfident for borderline
    is_borderline = 560 <= score <= 680
    if is_borderline:
        confidence = round(random.uniform(0.78, 0.88), 2)  # miscalibrated high
    else:
        abs_signal = min(1.0, abs(approve_score))
        confidence = round(0.75 + abs_signal * 0.2 + random.uniform(-0.05, 0.05), 2)

    reasoning = (
        f"Credit score {score} {'supports' if score > 650 else 'concerns'} approval. "
        f"DTI ratio {dti:.0%} is {'acceptable' if dti < 0.40 else 'high'}. "
        f"Income level {'adequate' if case.monthly_income_vnd > 20_000_000 else 'borderline'}. "
        f"Fraud signal {fraud:.0%} {'minimal' if fraud < 0.1 else 'requires attention'}. "
        f"Overall risk assessment: {recommendation}."
    )
    return {'recommendation': recommendation, 'confidence': confidence, 'reasoning': reasoning}


# ── Run LLM-as-Judge on 20 golden cases ─────────────────────────────────────
judge  = MockJudgeAgent()
scores: List[JudgeScore] = []
for case in golden_ds[:20]:
    output = mock_risk_agent(case)
    scores.append(judge.judge(case, output))

# Aggregate scores
criteria = ['logical_consistency','factor_coverage','confidence_calib','actionability','conciseness']
avgs     = {c: statistics.mean(getattr(s, c) for s in scores) for c in criteria}
overall_avg = statistics.mean(s.overall for s in scores)
all_weaknesses = [w for s in scores for w in s.weaknesses]
weakness_count = defaultdict(int)
for w in all_weaknesses: weakness_count[w] += 1

print("\n" + "═"*55)
print(" LLM-as-Judge — RiskAgent Evaluation (20 cases)")
print("═"*55)
bar_scale = 10
for c, avg in avgs.items():
    bar = '█' * int(avg * bar_scale / 5)
    icon = '✅' if avg >= 4.0 else ('⚠️' if avg >= 3.0 else '❌')
    print(f"  {icon} {c.replace('_',' ').title():<26} {bar:<10} {avg:.2f}/5")
print(f"  {'─'*45}")
print(f"  {'Overall Score':<28} {'█'*int(overall_avg*bar_scale/5):<10} {overall_avg:.2f}/5")
print(f"\n Top Weaknesses Detected:")
for w, cnt in sorted(weakness_count.items(), key=lambda x: -x[1]):
    print(f"   [{cnt:2d}x] {w}")
print("\n → Action: Improve confidence calibration for borderline cases")

---
## Section 3: Regression Suite — Automated Quality Gates

In [ ]:
@dataclass
class RegressionResult:
    passed: bool
    accuracy: float
    threshold: float
    total: int
    correct: int
    false_positives: int    # approve bad loan
    false_negatives: int    # reject good loan
    tag_accuracy: Dict[str, float]
    failures: List[dict]


class RegressionSuite:
    """Automated regression test suite for LoanBot."""

    def __init__(self, golden: List[GoldenCase], pipeline_fn):
        self.golden      = golden
        self.pipeline_fn = pipeline_fn

    def run(self, accuracy_threshold: float = 0.94,
            fpr_threshold: float = 0.03,
            fnr_threshold: float = 0.05) -> RegressionResult:

        results        = []
        false_pos      = 0
        false_neg      = 0
        tag_correct    = defaultdict(int)
        tag_total      = defaultdict(int)

        for case in self.golden:
            output  = self.pipeline_fn(case)
            pred    = output['recommendation']
            correct = pred == case.expert_label

            if not correct:
                if pred == 'APPROVE' and case.expert_label == 'REJECT':
                    false_pos += 1
                else:
                    false_neg += 1

            for tag in case.tags:
                tag_total[tag] += 1
                if correct: tag_correct[tag] += 1

            results.append({'case_id': case.case_id, 'correct': correct,
                            'pred': pred, 'expected': case.expert_label,
                            'tags': case.tags})

        n       = len(results)
        n_right = sum(1 for r in results if r['correct'])
        accuracy = n_right / n

        n_reject  = sum(1 for c in self.golden if c.expert_label == 'REJECT')
        n_approve = sum(1 for c in self.golden if c.expert_label == 'APPROVE')
        fpr       = false_pos / max(n_reject, 1)
        fnr       = false_neg / max(n_approve, 1)

        all_pass = accuracy >= accuracy_threshold and fpr <= fpr_threshold and fnr <= fnr_threshold

        return RegressionResult(
            passed=all_pass, accuracy=accuracy,
            threshold=accuracy_threshold, total=n, correct=n_right,
            false_positives=false_pos, false_negatives=false_neg,
            tag_accuracy={t: tag_correct[t]/tag_total[t] for t in tag_total},
            failures=[r for r in results if not r['correct']]
        )

    def print_report(self, result: RegressionResult):
        status = '✅ PASSED' if result.passed else '❌ FAILED'
        print(f"\n{'═'*55}")
        print(f" Regression Suite Report — {status}")
        print(f"{'─'*55}")
        print(f" Overall Accuracy: {result.accuracy:.1%} (threshold: {result.threshold:.0%})")
        n_reject  = sum(1 for c in self.golden if c.expert_label == 'REJECT')
        n_approve = sum(1 for c in self.golden if c.expert_label == 'APPROVE')
        fpr = result.false_positives / max(n_reject, 1)
        fnr = result.false_negatives / max(n_approve, 1)
        print(f" False Positive Rate (FPR): {fpr:.1%} (threshold: 3%)")
        print(f" False Negative Rate (FNR): {fnr:.1%} (threshold: 5%)")
        print(f" Total: {result.total} | Correct: {result.correct} | FP: {result.false_positives} | FN: {result.false_negatives}")
        print(f"\n Accuracy by Tag (model weakness analysis):")
        for tag, acc in sorted(result.tag_accuracy.items(), key=lambda x: x[1]):
            bar = '█' * int(acc * 20)
            icon = '✅' if acc >= 0.94 else ('⚠️' if acc >= 0.85 else '❌')
            print(f"  {icon} {tag:<20} {bar:<20} {acc:.1%}")
        if result.failures:
            print(f"\n Failures ({len(result.failures)}):")
            for f in result.failures[:5]:
                print(f"  {f['case_id']}: expected {f['expected']}, got {f['pred']} | tags={f['tags']}")
        print(f"{'═'*55}")


# ── Test v3.0 pipeline ─────────────────────────────────────────────────────
def loanbot_v3_pipeline(case: GoldenCase) -> dict:
    return mock_risk_agent(case)

suite    = RegressionSuite(golden_ds, loanbot_v3_pipeline)
result   = suite.run(accuracy_threshold=0.94)
suite.print_report(result)

---
## Section 4: A/B Test Analyzer — Statistical Significance

In [ ]:
@dataclass
class ABTestResult:
    metric: str
    control_value: float
    treatment_value: float
    control_n: int
    treatment_n: int
    p_value: float
    significant: bool
    effect_size: float
    recommendation: str


class ABTestAnalyzer:
    """Statistical analysis for A/B testing LoanBot versions."""

    def chi_squared_test(self, k1: int, n1: int, k2: int, n2: int) -> float:
        """Chi-squared test for difference in proportions.
        k = successes, n = total. Returns p-value (approximate).
        """
        p1 = k1 / n1
        p2 = k2 / n2
        p_pool = (k1 + k2) / (n1 + n2)
        if p_pool == 0 or p_pool == 1:
            return 1.0
        se = math.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
        if se == 0:
            return 1.0
        z = abs(p1 - p2) / se
        # Approximate p-value from z-score (two-tailed)
        # Using approximation: p ≈ 2 * (1 - Phi(z))
        p = 2 * (1 - self._norm_cdf(z))
        return round(p, 4)

    def _norm_cdf(self, z: float) -> float:
        """Abramowitz and Stegun approximation of normal CDF."""
        t = 1 / (1 + 0.2316419 * z)
        poly = t * (0.319381530 + t * (-0.356563782
               + t * (1.781477937 + t * (-1.821255978 + t * 1.330274429))))
        return 1 - (1 / math.sqrt(2 * math.pi)) * math.exp(-z**2 / 2) * poly

    def cohens_h(self, p1: float, p2: float) -> float:
        """Cohen's h effect size for proportions."""
        return 2 * abs(math.asin(math.sqrt(p1)) - math.asin(math.sqrt(p2)))

    def analyze(self, metric: str,
                control_successes: int, control_n: int,
                treatment_successes: int, treatment_n: int) -> ABTestResult:

        p_control   = control_successes   / control_n
        p_treatment = treatment_successes / treatment_n
        p_value     = self.chi_squared_test(
            control_successes, control_n,
            treatment_successes, treatment_n
        )
        h     = self.cohens_h(p_control, p_treatment)
        sig   = p_value < 0.05
        delta = p_treatment - p_control

        if sig and delta > 0:
            rec = f"ROLLOUT treatment (significantly better by {delta:+.1%})"
        elif sig and delta < 0:
            rec = f"ROLLBACK treatment (significantly worse by {delta:+.1%})"
        else:
            rec = "CONTINUE test (not significant yet)"

        return ABTestResult(
            metric=metric,
            control_value=p_control, treatment_value=p_treatment,
            control_n=control_n, treatment_n=treatment_n,
            p_value=p_value, significant=sig,
            effect_size=round(h, 3), recommendation=rec
        )

    def print_report(self, results: List[ABTestResult]):
        print("\n" + "═"*65)
        print(" A/B Test Report — LoanBot v2 (Control) vs v3.0 (Treatment)")
        print("═"*65)
        for r in results:
            sig_tag = '✅ SIG' if r.significant else '⏳ n.s.'
            delta   = r.treatment_value - r.control_value
            arrow   = '↑' if delta > 0 else '↓'
            print(f"\n  {r.metric}")
            print(f"    Control   : {r.control_value:.1%}  (n={r.control_n:,})")
            print(f"    Treatment : {r.treatment_value:.1%}  (n={r.treatment_n:,})  {arrow} {abs(delta):.1%}")
            print(f"    p-value   : {r.p_value:.4f}  [{sig_tag}]  |  Cohen's h = {r.effect_size:.3f}")
            print(f"    → {r.recommendation}")
        print("\n" + "═"*65)


# ── Simulate A/B test results ──────────────────────────────────────────────
analyzer = ABTestAnalyzer()

ab_results = [
    # Decision Accuracy
    analyzer.analyze('Decision Accuracy',
                     int(45_000 * 0.892), 45_000,   # v2: 89.2%
                     int(5_000  * 0.947), 5_000),    # v3: 94.7%

    # False Positive Rate (lower is better — we flip so lower = "success")
    # Treat: 1 - FPR as "clean rate"
    analyzer.analyze('Clean Rate (1 - FPR)',
                     int(45_000 * (1 - 0.058)), 45_000,  # v2 FPR 5.8%
                     int(5_000  * (1 - 0.023)), 5_000),  # v3 FPR 2.3%

    # HITL Avoidance Rate (1 - HITL rate)
    analyzer.analyze('HITL Avoidance Rate',
                     int(45_000 * (1 - 0.012)), 45_000,  # v2 HITL 1.2%
                     int(5_000  * (1 - 0.003)), 5_000),  # v3 HITL 0.3%
]

analyzer.print_report(ab_results)

---
## Section 5: RAGAS Metrics — Faithfulness, Relevancy, Precision, Recall

In [ ]:
@dataclass
class RAGASResult:
    faithfulness: float         # claims grounded in context
    answer_relevancy: float     # output addresses the question
    context_precision: float    # relevant data fraction
    context_recall: float       # key factors mentioned

    @property
    def ragas_score(self) -> float:
        return statistics.harmonic_mean([
            self.faithfulness, self.answer_relevancy,
            self.context_precision, self.context_recall
        ])


class RAGASEvaluator:
    """Simplified RAGAS evaluation for LoanBot RiskAgent outputs."""

    def evaluate(self, case: GoldenCase, agent_output: dict,
                 context: dict) -> RAGASResult:
        reasoning   = agent_output.get('reasoning', '')
        prediction  = agent_output.get('recommendation', 'REVIEW')

        # Faithfulness: does reasoning only claim what context supports?
        # Check: does reasoning mention score close to actual score?
        actual_score = str(case.credit_score)
        score_mentioned = any(s in reasoning for s in [
            str(case.credit_score), str(case.credit_score - 1), str(case.credit_score + 1)
        ])
        faithfulness = 0.95 if score_mentioned else 0.75
        faithfulness += random.uniform(-0.05, 0.05)

        # Answer Relevancy: does output address "should we approve?"
        has_recommendation = prediction in ['APPROVE', 'REJECT']
        answer_relevancy = 0.95 if has_recommendation else 0.60
        answer_relevancy += random.uniform(-0.03, 0.03)

        # Context Precision: fraction of referenced data that's actually relevant
        key_fields   = ['credit_score', 'dti_ratio', 'monthly_income_vnd', 'fraud_probability']
        used_fields  = sum(1 for f in key_fields if f.split('_')[0] in reasoning.lower())
        context_precision = used_fields / len(key_fields)
        context_precision += random.uniform(-0.05, 0.08)

        # Context Recall: were all key factors mentioned?
        key_terms = ['credit', 'dti', 'income', 'fraud']
        recalled  = sum(1 for t in key_terms if t in reasoning.lower())
        context_recall = recalled / len(key_terms)
        context_recall += random.uniform(-0.03, 0.05)

        return RAGASResult(
            faithfulness=round(min(1.0, max(0.0, faithfulness)), 3),
            answer_relevancy=round(min(1.0, max(0.0, answer_relevancy)), 3),
            context_precision=round(min(1.0, max(0.0, context_precision)), 3),
            context_recall=round(min(1.0, max(0.0, context_recall)), 3),
        )


# ── Run RAGAS on 30 golden cases ─────────────────────────────────────────
ragas_eval = RAGASEvaluator()
ragas_scores: List[RAGASResult] = []

for case in golden_ds[:30]:
    output  = mock_risk_agent(case)
    context = {'credit_score': case.credit_score, 'dti_ratio': case.dti_ratio,
               'monthly_income_vnd': case.monthly_income_vnd,
               'fraud_probability': case.fraud_probability}
    ragas_scores.append(ragas_eval.evaluate(case, output, context))

metrics = {
    'Faithfulness':      statistics.mean(r.faithfulness      for r in ragas_scores),
    'Answer Relevancy':  statistics.mean(r.answer_relevancy  for r in ragas_scores),
    'Context Precision': statistics.mean(r.context_precision for r in ragas_scores),
    'Context Recall':    statistics.mean(r.context_recall    for r in ragas_scores),
    'RAGAS Score':       statistics.mean(r.ragas_score       for r in ragas_scores),
}
targets = {
    'Faithfulness': 0.90, 'Answer Relevancy': 0.92,
    'Context Precision': 0.85, 'Context Recall': 0.88, 'RAGAS Score': 0.87
}

print("\n" + "═"*55)
print(" RAGAS Evaluation — RiskAgent (30 cases)")
print("═"*55)
for name, val in metrics.items():
    target = targets[name]
    icon   = '✅' if val >= target else '⚠️'
    bar    = '█' * int(val * 20)
    sep    = '─' * 5 if name == 'RAGAS Score' else ''
    if sep: print(f"  {sep}")
    print(f"  {icon} {name:<22} {bar:<20} {val:.3f}  (target: {target:.2f})")
print("═"*55)

---
## Section 6: Continuous Improvement Dashboard

In [ ]:
class ImprovementDashboard:
    """Weekly evaluation dashboard for LoanBot continuous improvement."""

    def __init__(self):
        self._history: List[Dict] = []

    def record_week(self, week: str, accuracy: float, fpr: float, fnr: float,
                    judge_score: float, ragas: float, tag_acc: Dict[str, float]):
        self._history.append({
            'week': week, 'accuracy': accuracy, 'fpr': fpr, 'fnr': fnr,
            'judge_score': judge_score, 'ragas': ragas, 'tag_acc': tag_acc
        })

    def print_trend(self):
        print("\n" + "═"*70)
        print(" LoanBot Continuous Improvement Dashboard — Weekly Trend")
        print("═"*70)
        print(f" {'Week':<10} {'Accuracy':>9} {'FPR':>7} {'FNR':>7} {'Judge':>7} {'RAGAS':>7} {'Status':>8}")
        print(f" {'─'*68}")
        for h in self._history:
            acc_ok  = h['accuracy'] >= 0.94
            fpr_ok  = h['fpr']      <= 0.03
            jud_ok  = h['judge_score'] >= 4.0
            status  = '✅ PASS' if (acc_ok and fpr_ok and jud_ok) else '⚠️ WATCH'
            print(f" {h['week']:<10} {h['accuracy']:>9.1%} {h['fpr']:>7.1%} {h['fnr']:>7.1%} "
                  f"{h['judge_score']:>7.2f} {h['ragas']:>7.3f} {status:>8}")

        # Improvement recommendations
        if len(self._history) >= 2:
            latest = self._history[-1]
            prev   = self._history[-2]
            print(f"\n {'─'*68}")
            print(f" Week-over-Week Changes (latest vs previous):")
            for key, label in [('accuracy','Accuracy'), ('fpr','FPR'), ('judge_score','Judge')]:
                delta = latest[key] - prev[key]
                if key == 'fpr':
                    arrow = '↓' if delta < 0 else '↑'
                    good  = delta < 0
                else:
                    arrow = '↑' if delta > 0 else '↓'
                    good  = delta > 0
                icon = '✅' if good else '⚠️'
                print(f"   {icon} {label:<12}: {arrow} {abs(delta):.2%}" if key != 'judge_score'
                      else f"   {icon} {label:<12}: {arrow} {abs(delta):.2f}")

        # Priority recommendations
        latest = self._history[-1]
        print(f"\n Priority Actions:")
        if latest['tag_acc'].get('self_employed', 1.0) < 0.85:
            print(f"   🎯 self_employed accuracy {latest['tag_acc']['self_employed']:.1%} < 85% → targeted improvement needed")
        if latest['fpr'] > 0.03:
            print(f"   🎯 FPR {latest['fpr']:.1%} > 3% → tighten approval criteria or retrain")
        if latest['judge_score'] < 4.0:
            print(f"   🎯 Judge score {latest['judge_score']:.2f} < 4.0 → improve explanation quality")
        print(f"   ✅ Goodhart protection: rotate 20% of golden dataset next month")
        print("═"*70)


# ── Simulate 6 weeks of improvement journey ──────────────────────────────
dashboard = ImprovementDashboard()

weeks = [
    # week, accuracy, fpr, fnr, judge, ragas, tag_acc
    ('2026-W26', 0.892, 0.058, 0.042, 3.82, 0.841, {'borderline': 0.79, 'self_employed': 0.71, 'good_credit': 0.97}),
    ('2026-W27', 0.907, 0.048, 0.038, 3.91, 0.855, {'borderline': 0.80, 'self_employed': 0.73, 'good_credit': 0.97}),
    ('2026-W28', 0.921, 0.038, 0.031, 3.98, 0.862, {'borderline': 0.83, 'self_employed': 0.76, 'good_credit': 0.98}),
    ('2026-W29', 0.935, 0.029, 0.028, 4.08, 0.871, {'borderline': 0.85, 'self_employed': 0.79, 'good_credit': 0.98}),
    ('2026-W30', 0.942, 0.024, 0.025, 4.12, 0.878, {'borderline': 0.86, 'self_employed': 0.82, 'good_credit': 0.98}),
    ('2026-W31', 0.951, 0.021, 0.022, 4.19, 0.886, {'borderline': 0.87, 'self_employed': 0.85, 'good_credit': 0.99}),
]

for w, acc, fpr, fnr, jud, ragas, tags in weeks:
    dashboard.record_week(w, acc, fpr, fnr, jud, ragas, tags)

dashboard.print_trend()

print("\n\n" + "═"*60)
print(" MODULE 12 LAB COMPLETE")
print("═"*60)
print(" ✅ Golden Dataset: 100 expert-labeled cases built")
print(" ✅ LLM-as-Judge: RiskAgent scored across 5 criteria")
print(" ✅ Regression Suite: automated quality gate implemented")
print(" ✅ A/B Test: v2 vs v3.0 analyzed with p-values")
print(" ✅ RAGAS: 4 metrics computed for RiskAgent")
print(" ✅ Dashboard: 6-week improvement trend tracked")
print(f"\n Key Finding: self_employed segment needs improvement")
print(f" Action: targeted prompt update + 20 new self_employed test cases")
print("═"*60)

---
## Tổng kết: Evaluation Engineering Framework

```
                    ┌───────────────────────────┐
                    │      EVALUATE             │
                    │  Golden Dataset (100 cases)│
                    │  LLM-as-Judge (5 criteria) │
                    │  RAGAS (4 metrics)         │
                    └───────────┬───────────────┘
                                │
                    ┌───────────▼───────────────┐
                    │      IDENTIFY             │
                    │  Tag breakdown analysis    │
                    │  Weakness: self_employed   │
                    │  Priority ranking          │
                    └───────────┬───────────────┘
                                │
                    ┌───────────▼───────────────┐
                    │   FIX (EDD approach)      │
                    │  Write test cases FIRST    │
                    │  Modify prompt/logic       │
                    │  Verify improvement        │
                    └───────────┬───────────────┘
                                │
                    ┌───────────▼───────────────┐
                    │   RE-EVALUATE             │
                    │  Regression suite CI/CD    │
                    │  A/B test significant?     │
                    │  ROLLOUT or ROLLBACK       │
                    └───────────────────────────┘
```

### LoanBot v3.0 Evaluation Scorecard

| Dimension | Metric | v2 | v3.0 | Target | Status |
|-----------|--------|----|------|--------|---------|
| Correctness | Decision Accuracy | 89.2% | 94.7% | ≥ 94% | ✅ Pass |
| Correctness | False Positive Rate | 5.8% | 2.3% | ≤ 3% | ✅ Pass |
| Helpfulness | Judge Score | 3.82 | 4.19 | ≥ 4.0 | ✅ Pass |
| Faithfulness | RAGAS Score | 0.841 | 0.886 | ≥ 0.87 | ✅ Pass |
| Segment | self_employed acc | 71% | 85% | ≥ 85% | ✅ Pass |

**Decision: ROLLOUT LoanBot v3.0 full traffic ✅**